# RQ2 — M4 vs M5 Claim Decomposition Study

**Contributor C | Analysis scaffold (no API calls)**

**Research question:** Does claim-level retrieval/verification (M5) improve evidence grounding compared with draft-level retrieval/verification (M4), and at what additional cost?

**Status:** Runner output schema pending confirmation. This notebook reads local CSV only. It does not run models or incur API charges.

In [2]:
import sys

print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("Jupyter is working!")

Python: 3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]
Interpreter: e:\code\course\EvidenceGrounded-MultiAgent\.venv\Scripts\python.exe
Jupyter is working!


In [3]:
import pandas as pd
import matplotlib
import numpy as np

print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("numpy:", np.__version__)

print("RQ2 analysis environment is ready!")

pandas: 3.1.0
matplotlib: 3.11.2
numpy: 2.4.6
RQ2 analysis environment is ready!


## 1. Experiment design

- Compare M4 and M5 on the **same example IDs and the same frozen AnswerDrafts**.
- Keep dataset split, model versions, retrieval corpus, retrieval budget, prompts (where applicable), and evaluation protocol documented.
- M4 and M5 may use different retrieval units; report evidence budget and total API cost to make the comparison interpretable.
- Do not interpret a metric as correctness without ground truth or an explicitly validated evaluator.
- Record missing/error cases separately; never treat them as zero factual claims.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("research/results/rq2_results.csv")
print("Expected input:", DATA_PATH)

## 2. Proposed Runner export schema (to confirm with E)

One row per `(example_id, method)` where method is `M4` or `M5`.

Required columns: `example_id`, `method`, `status`, `draft_id`, `grounding_score`, `input_tokens`, `output_tokens`, `cost_usd`, `latency_seconds`.

Optional: `claim_count`, `supported_claim_count`, `unsupported_claim_count`, `retrieval_calls`, `error_type`, `model_id`, `dataset_split`, `retrieval_top_k`.

**Important:** `grounding_score` is a placeholder for a consistently defined evaluation metric, not necessarily accuracy. Confirm its meaning with E/D. For M4, claim-level counts may be unavailable; leave them missing rather than inventing zeros.

In [ ]:
required = {"example_id", "method", "status", "draft_id", "grounding_score", "input_tokens", "output_tokens", "cost_usd", "latency_seconds"}
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if not set(df["method"].dropna().unique()).issubset({"M4", "M5"}):
        raise ValueError("method must be M4 or M5")
    if df.duplicated(["example_id", "method"]).any():
        raise ValueError("Duplicate (example_id, method) rows")
    print(f"Loaded {len(df)} rows")
else:
    df = None
    print("No Runner export yet. Analysis cells will wait for a CSV; no synthetic data is used as evidence.")

## 3. Pairing and quality checks

Compare only matched examples where both methods succeeded. Separately report the failure counts. Verify `draft_id` equality to ensure the same base answer.

In [ ]:
if df is not None:
    print("Status counts:")
    display(df.groupby(["method", "status"], dropna=False).size().rename("n").reset_index())
    good = df[df["status"].eq("success")].copy()
    paired = good.pivot(index="example_id", columns="method")
    matched = paired.dropna(subset=[("grounding_score", "M4"), ("grounding_score", "M5")]).copy()
    if not matched.empty and not (matched[("draft_id", "M4")].astype(str) == matched[("draft_id", "M5")].astype(str)).all():
        raise ValueError("M4/M5 draft_id mismatch: compare identical base drafts")
    print("Matched successful examples:", len(matched))

## 4. Paired outcome and cost comparison

Use per-example paired differences (`M5 − M4`). Positive grounding difference indicates improvement **only if higher grounding_score is defined as better**. Positive cost/latency differences mean extra resource use.

In [ ]:
metrics = ["grounding_score", "input_tokens", "output_tokens", "cost_usd", "latency_seconds"]
if df is not None and not matched.empty:
    comparison = pd.DataFrame(index=matched.index)
    for metric in metrics:
        comparison[f"{metric}_M4"] = pd.to_numeric(matched[(metric, "M4")], errors="coerce")
        comparison[f"{metric}_M5"] = pd.to_numeric(matched[(metric, "M5")], errors="coerce")
        comparison[f"{metric}_delta"] = comparison[f"{metric}_M5"] - comparison[f"{metric}_M4"]
    display(comparison[[f"{m}_delta" for m in metrics]].describe().T)
else:
    comparison = None

## 5. Visualizations

In [ ]:
if comparison is not None:
    for metric, ylabel in [("grounding_score", "M5 - M4 grounding score"), ("cost_usd", "M5 - M4 cost (USD)"), ("latency_seconds", "M5 - M4 latency (s)")]:
        vals = comparison[f"{metric}_delta"].dropna()
        if len(vals):
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.hist(vals, bins=min(12, max(1, len(vals))))
            ax.axvline(0, linestyle="--")
            ax.set_xlabel(ylabel)
            ax.set_ylabel("Number of examples")
            ax.set_title(f"Paired difference: {metric}")
            plt.show()

## 6. Qualitative error analysis (manual annotation)

For selected examples, annotate claim omissions, over-decomposition, incorrect entity/date preservation, retrieval failures, and verifier disagreements. Store annotation separately from automatic metrics.

| example_id | method | observation | suspected cause | evidence |
|---|---|---|---|---|
| TBD | M5 | TBD | TBD | TBD |

## 7. Conclusions (fill after real experiments)

- **Effectiveness:** TBD; report paired effect and sample size.
- **Costs:** TBD; report token and USD overhead.
- **Failure rates:** TBD; do not silently exclude failures from overall reporting.
- **Limitations:** evaluator validity, retrieval budget differences, dataset size, and model variability.

**Next dependency:** Confirm Runner export fields, M4/M5 implementation, and scoring semantics with E.